In [ ]:
from typing import Callable, Tuple
import jax
import jax.numpy as jnp

from typing import Callable
import jax
import jax.numpy as jnp


class Manifold:
    def __init__(
        self,
        param_dim: int,
        ambient_dim: int,
        embedding_func: Callable[[jnp.ndarray], jnp.ndarray],
    ):
        """
        Manifold embedded in R^n.

        Args:
            param_dim: Dimension of the parameter space (1D curve or 2D surface).
            ambient_dim: Dimension of the ambient space (2D or 3D).
            embedding_func: Function f: R^{param_dim} -> R^{ambient_dim}
                that defines the manifold embedding.
        """
        self.param_dim = param_dim
        self.ambient_dim = ambient_dim
        self.embedding_func = embedding_func

    def embed(self, params: jnp.ndarray) -> jnp.ndarray:
        """
        Evaluate the embedding for a batch of parameter points.

        Args:
            params: Array of shape (..., param_dim) — parameter points.

        Returns:
            Array of shape (..., ambient_dim) — embedded points.
        """
        params = jnp.atleast_2d(params)
        return jax.vmap(self.embedding_func)(params)


# Example: 2D manifold in R^3 — a surface

# Define the embedding function
def surface_embedding(params):
    x, y = params
    z = jnp.cos(x + y) / 2 + jnp.sin(x - y) / 2 + 1
    return jnp.array([x, y, z])

# Instantiate the manifold
manifold = Manifold(
    param_dim=2,
    ambient_dim=3,
    embedding_func=surface_embedding,
)

# Create a grid in parameter space
x = jnp.linspace(-jnp.pi, jnp.pi, 100)
y = jnp.linspace(-jnp.pi, jnp.pi, 100)
X, Y = jnp.meshgrid(x, y)

# Flatten grid for evaluation
param_points = jnp.stack([X.flatten(), Y.flatten()], axis=-1)

# Evaluate the manifold
embedded_points = manifold.embed(param_points)

# Reshape to grid shape
Z = embedded_points[:, 2].reshape(X.shape)

# Plot the surface
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(X, Y, Z, cmap='viridis', alpha=0.8)
plt.show()
